# Low-data multimodal pipeline: BUSI & COVID-19
global ViT feature + local small-kernel branch + LLM text

Run the cells top to bottom, in Google Colab (T4 GPU) or locally (CUDA or ROCm GPU). Everything expensive is cached
on Google Drive, so after a disconnect just re-run the cells and it resumes.

Tokens needed: HF_TOKEN (MedGemma access), KAGGLE_API_TOKEN (dataset download).
On Colab put them in Colab Secrets; locally put them in a .env file (any parent folder of this notebook),
which is loaded automatically. Never paste tokens into a cell.

Phases:  1 baseline | 2 local branch | 3 text (caption -> audit -> embed) | 4 fusion | 5 evaluation

In [1]:
# CELL 0 - Setup and config
import os, re, json, time, glob, gc, random, hashlib, subprocess, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
import sys
IN_COLAB = "google.colab" in sys.modules

# local runs: load HF_TOKEN / KAGGLE_API_TOKEN from the nearest .env found walking up from the working dir
def _load_env():
    d = os.path.abspath(os.getcwd())
    while True:                                       # walk up to the nearest .env (no python-dotenv needed)
        f = os.path.join(d, ".env")
        if os.path.isfile(f):
            for line in open(f, encoding="utf8"):
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip().removeprefix("export ").strip(), v.strip().strip("'\""))
            return f
        if os.path.dirname(d) == d: return None
        d = os.path.dirname(d)
if not IN_COLAB:
    print(".env loaded from:", _load_env())

if IN_COLAB:   # locally, install requirements yourself (see README / setup notes)
    get_ipython().system("pip install -q -U transformers bitsandbytes accelerate timm open_clip_torch kaggle scikit-learn")

# ---------------- experiment config ----------------
DATASET = "COVID"                            # Toggle: "BUSI" or "COVID"
SEED = 42                                   # train/test split seed (fixed)
FRACTIONS = [0.0063, 0.0125, 0.125, 1.0]    # 0.63%, 1.25%, 12.5%, 100% per roadmap
TEXT_FRACTIONS = [0.0063, 0.0125, 0.125]    # text variants run only here (needs far fewer captions)
SUBSET_SEEDS = [0, 1, 2, 3, 4]              # default number of resampled low-data subsets
SEEDS_FOR = {0.0063: [0, 1, 2, 3, 4], 0.0125: [0, 1, 2, 3, 4], 0.125: [0, 1, 2], 1.0: [0, 1]}  # fewer repeats where variance is small
POOL_FRAC = 0.25                            # share of train that gets captioned; low-data subsets are drawn from this pool
CAPTION_FULL_TRAIN = False                  # True = caption ALL train images (needed for text results at 100%)
EPOCHS, BATCH = 100, 8
BIG_BATCH = 32                              # used when a train subset has >=128 images (fewer, larger steps)
CAP_BATCH = 1                               # images per generate() call; on a 12 GB RX 6700 XT, 2 spills past VRAM with no speedup
LR_HEAD, LR_MODULES = 1e-4, 1e-3            # asynchronous LRs
HID, MID_DIM, K_SMALL = 128, 16, 5          # fusion width, local bottleneck, small kernel
P_DROP, P_MOD = 0.3, 0.2                    # dropout, modality dropout

# ---------------- dataset-specific config ----------------
if DATASET == "BUSI":
    MODALITY = "breast ultrasound"
    MAX_PER_CLASS = None                    # BUSI is small: use every image
    KAGGLE_DS = "aryashah2k/breast-ultrasound-images-dataset"
    HARD_LEAK = re.compile(r"benign|malignan|cancer|carcinom|bi-?rads|neoplas", re.I)
    SOFT_FLAG = re.compile(r"\bnormal\b|tumou?r", re.I)
    QUESTIONS = [
        "Is a distinct lesion visible? If so, is its shape oval, round, or irregular? If none, say so.",
        "Describe the lesion margin (circumscribed, indistinct, angular, microlobulated, spiculated), or say no lesion.",
        "Is the lesion orientation parallel or not parallel to the skin? If no lesion, say 'no lesion'.",
        "Describe the internal echo pattern (anechoic, hypoechoic, isoechoic, hyperechoic, heterogeneous).",
        "Describe posterior acoustic features (enhancement, shadowing, none). If no lesion, say 'no lesion'.",
        "Are calcifications or echogenic foci visible?",
        "Describe the surrounding tissue (fat, glandular, architectural distortion).",
        "Where is the lesion in the image, and roughly how large relative to the image? If no lesion, say 'no lesion'.",
        "Is there one lesion or multiple?",
        "Describe acoustic artifacts only (e.g., reverberation, poor penetration, speckle). Ignore any text, calipers, or markers."
    ]
elif DATASET == "COVID":
    MODALITY = "chest X-ray"
    MAX_PER_CLASS = 500                     # full COVID (~21k imgs) will not fit Colab RAM / captioning time; None disables
    KAGGLE_DS = "tawsifurrahman/covid19-radiography-database"
    HARD_LEAK = re.compile(r"covid|sars|coronavirus|pneumoni", re.I)
    SOFT_FLAG = re.compile(r"\bnormal\b|opacit|viral", re.I)
    QUESTIONS = [
        "Are there visible opacities or consolidations? If so, are they bilateral, peripheral, or lower-zone predominant?",
        "Are there ground-glass opacities (GGO)?",
        "Is there evidence of pleural effusion?",
        "Are the lung volumes normal, reduced, or hyperinflated?",
        "Is the cardiac silhouette normal in size and shape?",
        "Are the hilar and mediastinal contours normal?",
        "Are there any nodules or masses visible?",
        "Is there evidence of pneumothorax?",
        "Describe any visible hardware (e.g., tubes, lines, pacemaker).",
        "Comment on image quality, patient positioning (AP/PA), or artifacts."
    ]

# ---------------- paths / environment ----------------
DRIVE_DIR = f"/content/drive/MyDrive/mm_pipeline_{DATASET.lower()}"
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
except Exception:
    DRIVE_DIR = f"./mm_pipeline_{DATASET.lower()}"
    userdata = None
os.makedirs(DRIVE_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("dataset:", DATASET, "| device:", DEVICE, "| outputs:", DRIVE_DIR)

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

.env loaded from: e:\Research\.env
dataset: COVID | device: cuda | outputs: ./mm_pipeline_covid


In [2]:
# CELL 1 - Data, labels, fixed split
def download_dataset():
    root = f"/content/Dataset_{DATASET}" if IN_COLAB else f"./Dataset_{DATASET}"
    if os.path.isdir(root):
        return root
    tok = secret("KAGGLE_API_TOKEN")
    assert tok, "Add KAGGLE_API_TOKEN to Colab Secrets (and revoke any token you pasted in a notebook)."
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    p = os.path.expanduser("~/.kaggle/access_token")
    open(p, "w").write(tok); os.chmod(p, 0o600)
    os.environ["KAGGLE_API_TOKEN"] = tok
    from kaggle.api.kaggle_api_extended import KaggleApi   # in-process: no dependency on the kaggle executable being on PATH
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files(KAGGLE_DS, path=root, unzip=True, quiet=False)
    return root

def load_dataset():
    root = download_dataset()
    paths = sorted(glob.glob(f"{root}/**/*.png", recursive=True) + glob.glob(f"{root}/**/*.jpg", recursive=True))
    paths = [p for p in paths if "mask" not in p.lower()]   # drop segmentation masks
    def _label(p):   # COVID layout is <Class>/images/<file>.png, BUSI is <class>/<file>.png
        d = os.path.basename(os.path.dirname(p))
        return os.path.basename(os.path.dirname(os.path.dirname(p))) if d.lower() == "images" else d
    d = pd.DataFrame({"image_path": paths, "label": [_label(p) for p in paths]})
    if MAX_PER_CLASS:   # keeps Colab RAM / captioning time manageable (see config)
        d = pd.concat([g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED) for _, g in d.groupby("label")])
        d = d.sort_values("image_path").reset_index(drop=True)
    return d

from sklearn.model_selection import train_test_split
SPLIT_CSV = f"{DRIVE_DIR}/split.csv"
raw = load_dataset()
if os.path.exists(SPLIT_CSV):
    df = pd.read_csv(SPLIT_CSV)
else:
    tr_, te_ = train_test_split(raw, test_size=0.2, stratify=raw["label"], random_state=SEED)
    df = pd.concat([tr_.assign(split="train"), te_.assign(split="test")], ignore_index=True)
    df.to_csv(SPLIT_CSV, index=False)
df["id"] = df["image_path"].apply(lambda p: hashlib.md5(p.encode()).hexdigest()[:10] + "_" + os.path.basename(p))
CLASSES = sorted(df["label"].unique())
df["y"] = df["label"].map({c: i for i, c in enumerate(CLASSES)})
NCLS, N = len(CLASSES), len(df)
train_idx_all = np.where(df["split"].values == "train")[0]
test_idx = np.where(df["split"].values == "test")[0]
print(CLASSES, "| total", N, "| train", len(train_idx_all), "| test", len(test_idx))
print(df.groupby(["split", "label"]).size().unstack())

['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia'] | total 2000 | train 1600 | test 400
label  COVID  Lung_Opacity  Normal  Viral Pneumonia
split                                              
test     100           100     100              100
train    400           400     400              400


In [3]:
# CELL 2 - Class-balanced, nested low-data subsets (5-fold CV equivalents)
# Caption pool: a class-stratified random POOL_FRAC of the training set (seeded). Subsets with frac <= POOL_FRAC are
# drawn from it, so every subset is still a random class-balanced sample of the train set, but only ~POOL_FRAC of
# train ever needs captions. Fractions above POOL_FRAC (e.g. 100%) use the whole train set.
_r = np.random.RandomState(SEED); _pool = []
for c in range(NCLS):
    full = train_idx_all[df["y"].values[train_idx_all] == c]
    _pool.append(_r.permutation(full)[:max(1, int(round(POOL_FRAC * len(full))))])
pool_idx = np.concatenate(_pool)

def subset(frac, seed):
    rng = np.random.RandomState(seed); out = []
    for c in range(NCLS):
        full = train_idx_all[df["y"].values[train_idx_all] == c]
        n_c = max(1, int(round(frac * len(full))))
        src_ids = pool_idx[df["y"].values[pool_idx] == c] if frac <= POOL_FRAC else full
        src_ids = src_ids[rng.permutation(len(src_ids))]
        out.append(src_ids[:min(n_c, len(src_ids))])
    return np.concatenate(out)

for f in FRACTIONS:
    print(f"{f*100:g}% ->", len(subset(f, 0)), "train images (seed 0)")

0.63% -> 12 train images (seed 0)
1.25% -> 20 train images (seed 0)
12.5% -> 200 train images (seed 0)
100% -> 1600 train images (seed 0)


In [4]:
# CELL 3 - Frozen ViT-B/16 features (global CLS + patch tokens), cached
import timm
from timm.data import resolve_data_config, create_transform
FEATS = f"{DRIVE_DIR}/vit_feats.pt"
if os.path.exists(FEATS):
    blob = torch.load(FEATS)
    cls_all, patch_all = blob["cls"], blob["patch"]
else:
    vit = timm.create_model("vit_base_patch16_224", pretrained=True).to(DEVICE).eval()
    tfm = create_transform(**resolve_data_config({}, model=vit))
    class DS(torch.utils.data.Dataset):
        def __len__(s): return N
        def __getitem__(s, i): return tfm(Image.open(df["image_path"][i]).convert("RGB"))
    cls_l, patch_l = [], []
    with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
        for x in tqdm(torch.utils.data.DataLoader(DS(), batch_size=32, num_workers=0 if os.name == "nt" else 2)):
            t = vit.forward_features(x.to(DEVICE))          # B,197,768
            cls_l.append(t[:, 0].half().cpu()); patch_l.append(t[:, 1:].half().cpu())
    cls_all, patch_all = torch.cat(cls_l), torch.cat(patch_l)
    torch.save({"cls": cls_all, "patch": patch_all}, FEATS)
    del vit; gc.collect(); torch.cuda.empty_cache()
print("cls", tuple(cls_all.shape), "patch", tuple(patch_all.shape))

data = {
    "G": cls_all.to(DEVICE), "P": patch_all.to(DEVICE),
    "D": torch.zeros(N, 512, device=DEVICE), "Q": torch.zeros(N, 512, device=DEVICE),
    "y": torch.as_tensor(df["y"].values, device=DEVICE).long(),
    "has_text": np.zeros(N, dtype=bool),
}

cls (2000, 768) patch (2000, 196, 768)


In [5]:
# CELL 4 - Models and training engine
from sklearn.metrics import accuracy_score, f1_score, recall_score

class LocalBranch(nn.Module):
    def __init__(s, d=768, mid=MID_DIM, out=HID, k=K_SMALL):
        super().__init__()
        s.down = nn.Linear(d, mid)
        s.dw = nn.Conv2d(mid, mid, k, padding=k // 2, groups=mid)
        s.act = nn.GELU(); s.up = nn.Linear(mid, out)
    def forward(s, p):
        x = s.down(p).transpose(1, 2).reshape(p.size(0), -1, 14, 14)
        return s.up(s.act(s.dw(x)).mean((2, 3)))

class LinearProbe(nn.Module):
    def __init__(s, n_cls):
        super().__init__(); s.fc = nn.Linear(768, n_cls)
    def forward(s, b): return s.fc(b["g"])

class Fusion(nn.Module):
    SRC = {"g": "g", "l": "p", "d": "d", "q": "q"}
    def __init__(s, n_cls, g=0, l=0, d=0, q=0):
        super().__init__()
        s.enc = nn.ModuleDict()
        if g: s.enc["g"] = nn.Sequential(nn.LayerNorm(768), nn.Linear(768, HID))
        if l: s.enc["l"] = LocalBranch()
        if d: s.enc["d"] = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, HID))
        if q: s.enc["q"] = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, HID))
        s.keys = list(s.enc.keys())
        s.head = nn.Sequential(nn.Dropout(P_DROP), nn.Linear(len(s.keys) * HID, HID), nn.GELU(), nn.Linear(HID, n_cls))
    def forward(s, b):
        z = torch.stack([s.enc[k](b[s.SRC[k]]) for k in s.keys], 1)
        if s.training and z.size(1) > 1 and P_MOD > 0:
            m = (torch.rand(z.size(0), z.size(1), device=z.device) > P_MOD).float()
            none = m.sum(1) == 0
            if none.any():
                r = torch.randint(0, z.size(1), (int(none.sum()),), device=z.device)
                m[none.nonzero().squeeze(1), r] = 1
            z = z * m.unsqueeze(-1)
        return s.head(z.flatten(1))

VARIANTS = {
    "linear_probe":       dict(kind="linear"),
    "global_mlp":         dict(g=1),
    "global+local":       dict(g=1, l=1),
    "global+desc":        dict(g=1, d=1),
    "global+qa":          dict(g=1, q=1),
    "full":               dict(g=1, l=1, d=1, q=1),
    "text_only":          dict(d=1, q=1),
    "full_shuffled_text": dict(g=1, l=1, d=1, q=1, shuffle=True),
}
needs_text = lambda spec: bool(spec.get("d") or spec.get("q"))

def make_batch(idx, tidx):
    return {"g": data["G"][idx].float(), "p": data["P"][idx].float(), "d": data["D"][tidx], "q": data["Q"][tidx]}

def train_eval(spec, tr_idx, te_idx, seed):
    set_seed(seed)
    model = (LinearProbe(NCLS) if spec.get("kind") == "linear"
             else Fusion(NCLS, **{k: spec.get(k, 0) for k in "glqd"})).to(DEVICE)
    if spec.get("kind") == "linear":
        groups = [{"params": model.parameters(), "lr": LR_HEAD}]
    else:
        head = list(model.head[-1].parameters()); hid = {id(p) for p in head}
        groups = [{"params": head, "lr": LR_HEAD},
                  {"params": [p for p in model.parameters() if id(p) not in hid], "lr": LR_MODULES}]
    opt = torch.optim.Adam(groups)
    tr_t, te_t = tr_idx.copy(), te_idx.copy()
    if spec.get("shuffle"):   # control: every image gets the text of a DIFFERENT, randomly chosen image
        rng = np.random.RandomState(seed + 999)
        def derange(a):
            if len(a) < 2: return a
            while True:
                p = a[rng.permutation(len(a))]
                if (p != a).all(): return p
        tr_t, te_t = derange(tr_t), derange(te_t)
    tr_i, tr_ti = torch.as_tensor(tr_idx, device=DEVICE), torch.as_tensor(tr_t, device=DEVICE)
    te_i, te_ti = torch.as_tensor(te_idx, device=DEVICE), torch.as_tensor(te_t, device=DEVICE)
    n = len(tr_idx); bs = BIG_BATCH if n >= 128 else BATCH
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            bi = perm[i:i + bs]
            loss = F.cross_entropy(model(make_batch(tr_i[bi], tr_ti[bi])), data["y"][tr_i[bi]])
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(make_batch(te_i, te_ti)).argmax(1).cpu().numpy()
    yt = data["y"][te_i].cpu().numpy()
    return dict(acc=100 * accuracy_score(yt, pred), f1=f1_score(yt, pred, average="macro"),
                sen=100 * recall_score(yt, pred, average="macro"),
                params=sum(p.numel() for p in model.parameters() if p.requires_grad))

RESULTS_CSV = f"{DRIVE_DIR}/results_fast.csv"   # new name: old subsets are no longer comparable
COLS = ["variant", "frac", "seed", "n_train", "acc", "f1", "sen", "params"]

def run_grid(names, fractions=FRACTIONS, seeds=None):
    res = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(columns=COLS)
    done = set(zip(res["variant"], res["frac"].round(4), res["seed"]))
    jobs = [(nm, f, s) for nm in names for f in fractions
            for s in (seeds or SEEDS_FOR.get(round(f, 4), SUBSET_SEEDS)) if (nm, round(f, 4), s) not in done]
    for nm, f, s in tqdm(jobs):
        spec, tr = VARIANTS[nm], subset(f, s)
        if needs_text(spec) and not (data["has_text"][tr].all() and data["has_text"][test_idx].all()):
            print(f"skip {nm} {f*100:g}% seed {s}: captions missing for needed images"); continue
        row = dict(variant=nm, frac=round(f, 4), seed=s, n_train=len(tr), **train_eval(spec, tr, test_idx, s))
        pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", header=not os.path.exists(RESULTS_CSV), index=False)  # resumable
    return pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(columns=COLS)

In [6]:
# CELL 5 - Phases 1-2: baseline, matched control, local branch
res = run_grid(["linear_probe", "global_mlp", "global+local"])
print(res.groupby(["variant", "frac"])[["acc", "f1", "sen"]].mean().round(2))

  0%|          | 0/45 [00:00<?, ?it/s]

                       acc    f1    sen
variant      frac                      
global+local 0.0063  56.40  0.55  56.40
             0.0125  62.90  0.63  62.90
             0.1250  78.75  0.79  78.75
             1.0000  86.38  0.86  86.38
global_mlp   0.0063  57.00  0.56  57.00
             0.0125  63.50  0.63  63.50
             0.1250  78.58  0.79  78.58
             1.0000  84.75  0.85  84.75
linear_probe 0.0063  55.20  0.54  55.20
             0.0125  60.70  0.60  60.70
             0.1250  75.33  0.75  75.33
             1.0000  85.00  0.85  85.00


In [7]:
# CELL 6 - Phase 3a: caption with MedGemma (resumable)
# Two ways to run the SAME model (MedGemma 4B-it) with the SAME prompt:
#   "llamacpp"     : local llama.cpp server, Q8_0 GGUF on the GPU via Vulkan (~11 s/image on an RX 6700 XT).
#                    The reply is forced to match a JSON schema, so it always parses.
#   "transformers" : Hugging Face transformers (Colab / NVIDIA): bf16, or 4-bit with bitsandbytes.
# Each backend writes its own caption file, so captions from the two are never mixed.
import urllib.request, base64, atexit

MID = "google/medgemma-4b-it"
BUDGET_MIN = 150          # per-session time budget; re-run this cell to continue
SYSTEM = "You are an expert radiologist."

TOOLS = os.path.abspath(os.path.join("..", "tools"))         # E:\Research\tools (not in git)
LLAMA_SERVER = os.path.join(TOOLS, "llama.cpp-b12383", "llama-server.exe")
GGUF_MODEL = os.path.join(TOOLS, "gguf", "medgemma-4b-it-Q8_0.gguf")
GGUF_MMPROJ = os.path.join(TOOLS, "gguf", "mmproj-F16.gguf")
LLAMA_PORT = 8089
BACKEND = "llamacpp" if (not IN_COLAB and all(map(os.path.exists, [LLAMA_SERVER, GGUF_MODEL, GGUF_MMPROJ]))) else "transformers"

def build_prompt():
    qs = "\n".join(f"{i+1}. {q}" for i, q in enumerate(QUESTIONS))
    return (f"This is a {MODALITY} image.\n"
            "Return ONLY valid JSON, nothing else, with exactly this structure:\n"
            '{"description": "<2-3 short sentences describing the visible findings>", '
            f'"answers": ["<answer 1>", ..., "<answer {len(QUESTIONS)}>"]}}\n'
            f"The list must contain exactly {len(QUESTIONS)} strings, one per question, each AT MOST 15 words:\n{qs}\n"
            "Do NOT state a diagnosis, disease name, or BI-RADS category.")
PROMPT = build_prompt()
# captions live in a file named after a hash of (model/backend + prompt): a changed prompt or backend never mixes with old captions
CAP_TAG = MID if BACKEND == "transformers" else f"llamacpp:{os.path.basename(GGUF_MODEL)}+{os.path.basename(GGUF_MMPROJ)}"
CAPS = f"{DRIVE_DIR}/captions_{hashlib.md5((CAP_TAG + PROMPT).encode()).hexdigest()[:8]}.jsonl"
print("backend:", BACKEND, "| caption file:", CAPS)

if BACKEND == "llamacpp":
    LLAMA_URL = f"http://127.0.0.1:{LLAMA_PORT}"
    QUANT = "Q8_0 GGUF + F16 mmproj (unsloth/medgemma-4b-it-GGUF), llama.cpp b12383, Vulkan"
    SCHEMA = {"type": "object", "required": ["description", "answers"],
              "properties": {"description": {"type": "string"},
                             "answers": {"type": "array", "items": {"type": "string"},
                                         "minItems": len(QUESTIONS), "maxItems": len(QUESTIONS)}}}

    def _llama_ok():
        try: return json.load(urllib.request.urlopen(LLAMA_URL + "/health", timeout=2)).get("status") == "ok"
        except Exception: return False

    def start_llama():
        global _llama_proc
        if _llama_ok():
            print("llama-server already running"); return
        log = open(f"{DRIVE_DIR}/llama_server.log", "w")
        _llama_proc = subprocess.Popen([LLAMA_SERVER, "-m", GGUF_MODEL, "--mmproj", GGUF_MMPROJ, "-ngl", "99",
                                        "-c", "4096", "-fa", "on", "--host", "127.0.0.1", "--port", str(LLAMA_PORT)],
                                       stdout=log, stderr=subprocess.STDOUT)
        for _ in range(240):
            if _llama_ok():
                print("llama-server ready"); return
            if _llama_proc.poll() is not None:
                raise RuntimeError(f"llama-server exited early; see {DRIVE_DIR}/llama_server.log")
            time.sleep(1)
        raise RuntimeError(f"llama-server not ready after 4 min; see {DRIVE_DIR}/llama_server.log")

    def stop_llama():                                       # frees the GPU memory (~6 GB)
        p = globals().get("_llama_proc")
        if p is not None and p.poll() is None:
            p.terminate(); p.wait(30); print("llama-server stopped")

    _llama_proc = None
    atexit.register(stop_llama)

    def _ask_one(path, max_new_tokens, sample, seed):
        img = base64.b64encode(open(path, "rb").read()).decode()
        mime = "image/png" if path.lower().endswith(".png") else "image/jpeg"
        body = {"messages": [{"role": "system", "content": SYSTEM},
                             {"role": "user", "content": [
                                 {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{img}"}},
                                 {"type": "text", "text": PROMPT}]}],
                "max_tokens": max_new_tokens, "cache_prompt": False, "seed": seed,
                "response_format": {"type": "json_schema", "json_schema": {"name": "caption", "schema": SCHEMA}}}
        body.update(dict(temperature=0.7, top_p=0.9) if sample else dict(temperature=0))
        req = urllib.request.Request(LLAMA_URL + "/v1/chat/completions", json.dumps(body).encode(),
                                     {"Content-Type": "application/json"})
        return json.load(urllib.request.urlopen(req, timeout=600))["choices"][0]["message"]["content"]

    def ask_batch(paths, max_new_tokens=450, sample=False, seed=0):
        return [_ask_one(p, max_new_tokens, sample, seed) for p in paths]

else:
    from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
    hf = secret("HF_TOKEN")
    assert hf, "HF_TOKEN is empty: put HF_TOKEN=... in your .env (or Colab Secrets) and re-run Cell 0"
    os.environ["HF_TOKEN"] = hf
    proc = AutoProcessor.from_pretrained(MID, token=hf)
    proc.tokenizer.padding_side = "left"                      # required for batched generation
    try:
        import accelerate; _kw = dict(device_map="auto", token=hf)
    except ImportError:                                       # no accelerate: load on CPU, then move to the GPU below
        print("accelerate not installed (pip install accelerate); loading without device_map")
        _kw = dict(token=hf)
    _use_bnb = False
    if torch.cuda.is_available() and not getattr(torch.version, "hip", None):   # bitsandbytes 4-bit: NVIDIA only
        try:
            import bitsandbytes; _use_bnb = True
        except ImportError:
            pass
    if _use_bnb:
        _kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:                                                     # ROCm / CPU: plain bf16 (~8 GB for 4B); fp16 overflows
        _kw["dtype"] = torch.bfloat16
    QUANT = "bnb 4-bit" if _use_bnb else "bf16"
    print("MedGemma load:", QUANT, "| device:", DEVICE)
    mg = AutoModelForImageTextToText.from_pretrained(MID, **_kw).eval()
    if "device_map" not in _kw: mg = mg.to(DEVICE)
    # dtype of the unquantized vision tower, so pixel_values match whatever transformers version is installed
    VDTYPE = next((p.dtype for n, p in mg.named_parameters() if "vision" in n and p.is_floating_point()), torch.float16)

    def _msgs(path):
        img = Image.open(path).convert("RGB")
        return [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
                {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]

    # The answer is pre-filled with the start of the JSON, so the model must answer in JSON straight away
    # (otherwise MedGemma 1.5 may write a long hidden "thought" first and run out of tokens).
    PREFILL = '{"description": "'

    @torch.inference_mode()
    def ask_batch(paths, max_new_tokens=450, sample=False, seed=0):
        msgs = [_msgs(p) for p in paths]
        texts = [proc.apply_chat_template(m, add_generation_prompt=True, tokenize=False) + PREFILL for m in msgs]
        imgs = [[m[-1]["content"][0]["image"]] for m in msgs]
        inp = proc(text=texts, images=imgs, return_tensors="pt", padding=True).to(mg.device)
        if "pixel_values" in inp: inp["pixel_values"] = inp["pixel_values"].to(VDTYPE)
        n = inp["input_ids"].shape[-1]
        if sample: torch.manual_seed(seed)
        kw = dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False)
        out = mg.generate(**inp, max_new_tokens=max_new_tokens, **kw)
        return [PREFILL + proc.decode(o[n:], skip_special_tokens=True) for o in out]

def parse_json(txt):
    txt = txt.split("<unused95>")[-1]                     # drop a "thinking" block if one slipped through
    t = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    d = json.loads(t[t.find("{"): t.rfind("}") + 1])
    ans = [str(a) for a in d["answers"]]
    assert len(ans) == len(QUESTIONS)
    return {"description": str(d["description"]), "answers": ans}

def to_record(raw, a=0):
    try: p = parse_json(raw)
    except Exception: return dict(raw=raw, parse_ok=False, attempts=a + 1)
    # leaks are only FLAGGED here; the hard-leak words are redacted before embedding (Cell 8), so no costly regeneration
    return dict(raw=raw, parse_ok=True, hard_leak=bool(HARD_LEAK.search(json.dumps(p))), attempts=a + 1, **p)

# queue: test set first, then exactly the train images that the text experiments will use
queue, seen = list(test_idx), set(test_idx)
for f in TEXT_FRACTIONS:
    for s in SEEDS_FOR.get(round(f, 4), SUBSET_SEEDS):
        for i in subset(f, s):
            if i not in seen: queue.append(i); seen.add(i)
if CAPTION_FULL_TRAIN:
    queue += [i for i in train_idx_all if i not in seen]

done = set()
if os.path.exists(CAPS):
    done = {r["id"] for r in map(json.loads, open(CAPS)) if r.get("parse_ok")}   # failed ones get retried
todo = [i for i in queue if df["id"][i] not in done]
print(f"{len(queue)} images needed | {len(done)} done | {len(todo)} to go")
if BACKEND == "llamacpp" and todo: start_llama()     # only start the server when there is work to do
t0 = time.time()
try:
    with open(CAPS, "a") as fh:
        for k in tqdm(range(0, len(todo), CAP_BATCH)):
            if time.time() - t0 > BUDGET_MIN * 60:
                print("time budget reached; re-run this cell to continue"); break
            chunk = todo[k:k + CAP_BATCH]; paths = [df["image_path"][i] for i in chunk]
            try:
                raws = ask_batch(paths)
            except Exception as e:                            # e.g. OOM: fall back to one image at a time
                print("batch failed, going image by image:", repr(e)[:120]); torch.cuda.empty_cache(); raws = []
                for p in paths:
                    try: raws.append(ask_batch([p])[0])
                    except Exception as e2: print("  failed:", os.path.basename(p), repr(e2)[:120]); raws.append("")
            for i, raw in zip(chunk, raws):
                rec, a = to_record(raw), 0
                while not rec["parse_ok"] and a < 2:           # retry ONLY unparseable outputs, sampled
                    a += 1
                    try: rec = to_record(ask_batch([df["image_path"][i]], sample=True, seed=a)[0], a)
                    except Exception: break
                fh.write(json.dumps({"id": df["id"][i], "label": df["label"][i], **rec}) + "\n"); fh.flush()
finally:
    if BACKEND == "llamacpp": stop_llama()                 # also runs if you interrupt the cell

json.dump({"model": MID, "backend": BACKEND, "quantization": QUANT, "prompt": PROMPT, "system": SYSTEM,
           "questions": QUESTIONS, "caps_file": CAPS, "pool_frac": POOL_FRAC,
           "text_fractions": TEXT_FRACTIONS, "cap_batch": CAP_BATCH},
          open(f"{DRIVE_DIR}/run_info.json", "w"), indent=2)

backend: llamacpp | caption file: ./mm_pipeline_covid/captions_5d6d7f4d.jsonl
746 images needed | 0 done | 746 to go
llama-server ready


  0%|          | 0/746 [00:00<?, ?it/s]

llama-server stopped


In [8]:
# CELL 7 - Phase 3b: audit
rows = {r["id"]: r for r in map(json.loads, open(CAPS))}
cap = pd.DataFrame.from_dict(rows, orient="index")
print("captioned:", len(cap), "| parse failures:", int((~cap["parse_ok"]).sum()))
ok = cap[cap["parse_ok"]].copy()
alltext = ok.apply(lambda r: r["description"] + " " + " ".join(r["answers"]), axis=1)
ok["hard"] = alltext.apply(lambda t: bool(HARD_LEAK.search(t)))
ok["soft"] = alltext.apply(lambda t: bool(SOFT_FLAG.search(t)))
print("\nleak flags by class (before redaction):"); print(ok.groupby("label")[["hard", "soft"]].sum())

def redact(t): return re.sub(r"\s+", " ", HARD_LEAK.sub("", t)).strip()
ok["clean"] = alltext.apply(redact)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
split_of = df.set_index("id")["split"]
trn, tst = ok[ok.index.map(split_of) == "train"], ok[ok.index.map(split_of) == "test"]
if len(trn) > 30 and len(tst) > 10:
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), LogisticRegression(max_iter=2000))
    pipe.fit(trn["clean"], trn["label"])
    print(f"\ntext-only TF-IDF accuracy: {100*pipe.score(tst['clean'], tst['label']):.1f}%  |  "
          f"majority baseline: {100*tst['label'].value_counts(normalize=True).iloc[0]:.1f}%  (>95% => suspect leakage)")

aud = ok.sample(min(50, len(ok)), random_state=0)[["label", "description"]].copy()
aud.insert(0, "image_path", [df.set_index("id").loc[i, "image_path"] for i in aud.index])
aud["rating_1to5"] = ""; aud["notes"] = ""
aud.to_csv(f"{DRIVE_DIR}/caption_audit_sheet.csv")

captioned: 746 | parse failures: 0

leak flags by class (before redaction):
                 hard  soft
label                      
COVID               3   189
Lung_Opacity       11   188
Normal              0   185
Viral Pneumonia    18   184

text-only TF-IDF accuracy: 60.8%  |  majority baseline: 25.0%  (>95% => suspect leakage)


In [9]:
# CELL 8 - Phase 3c: embed text (BiomedCLIP)
if "stop_llama" in globals(): stop_llama()          # free the GPU if Cell 6 left llama-server running
for _n in ("mg", "proc"): globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()
import open_clip
BIOMED = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
bclip, _ = open_clip.create_model_from_pretrained(BIOMED)
btok = open_clip.get_tokenizer(BIOMED)
bclip = bclip.to(DEVICE).eval()

@torch.no_grad()
def embed(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        t = btok(texts[i:i + bs], context_length=256).to(DEVICE)
        out.append(F.normalize(bclip.encode_text(t).float(), dim=-1).cpu())
    return torch.cat(out)

ids = list(ok.index)
desc_e = embed([redact(ok.loc[i, "description"]) for i in ids])
flat = [redact(a) for i in ids for a in ok.loc[i, "answers"]]
qa_e = embed(flat).reshape(len(ids), len(QUESTIONS), -1).mean(1)
pos = {i: k for k, i in enumerate(df["id"])}
for k, i in enumerate(ids):
    j = pos[i]
    data["D"][j] = desc_e[k].to(DEVICE); data["Q"][j] = F.normalize(qa_e[k], dim=0).to(DEVICE)
    data["has_text"][j] = True
del bclip; gc.collect(); torch.cuda.empty_cache()

In [10]:
# CELL 9 - Phase 4: Fusion runs
res = run_grid(["global+desc", "global+qa", "full", "text_only", "full_shuffled_text"], fractions=TEXT_FRACTIONS)

  0%|          | 0/65 [00:00<?, ?it/s]

In [11]:
# CELL 10 - Phase 5: results table & figure
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
res = pd.read_csv(RESULTS_CSV)
order = [v for v in VARIANTS if v in set(res["variant"])]
fr = sorted(res["frac"].unique())
for metric in ["acc", "f1", "sen"]:
    g = res.groupby(["variant", "frac"])[metric].agg(["mean", "std"])
    tab = g.apply(lambda r: f"{r['mean']:.2f}±{r['std']:.2f}" if metric != "f1" else f"{r['mean']:.3f}±{r['std']:.3f}", axis=1).unstack("frac")
    tab = tab.reindex(order)[fr]; tab.columns = [f"{c*100:g}%" for c in fr]
    print(f"\n=== {metric.upper()} ({DATASET}) ===")
    print(tab.to_string()); tab.to_csv(f"{DRIVE_DIR}/table_{metric}.csv")

fig, ax = plt.subplots(figsize=(8, 5))
for v in order:
    s = res[res.variant == v].groupby("frac")["acc"].agg(["mean", "std"]).reindex(fr)
    ax.errorbar(range(len(fr)), s["mean"], yerr=s["std"], marker="o", capsize=2, label=v,
                ls="--" if "shuffled" in v or v == "linear_probe" else "-")
ax.set_xticks(range(len(fr))); ax.set_xticklabels([f"{c*100:g}%" for c in fr])
ax.set_xlabel("Training set (%)"); ax.set_ylabel("Test ACC (%)")
ax.set_title(f"{DATASET}: low-data multimodal variants")
ax.legend(fontsize=8); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(f"{DRIVE_DIR}/acc_vs_fraction.png", dpi=200)


=== ACC (COVID) ===
                         0.63%        1.25%       12.5%        100%
variant                                                            
linear_probe        55.20±5.09   60.70±5.33  75.33±0.52  85.00±0.35
global_mlp          57.00±4.99   63.50±5.95  78.58±1.28  84.75±1.41
global+local        56.40±6.41   62.90±7.63  78.75±1.98  86.38±0.18
global+desc         54.05±3.50   60.05±5.69  79.08±0.14         NaN
global+qa           54.40±5.21  62.30±10.12  82.50±2.46         NaN
full                51.25±3.81   60.45±5.19  81.50±1.95         NaN
text_only           44.25±3.04   44.35±1.86  55.75±1.50         NaN
full_shuffled_text  46.55±6.79   54.90±4.66  77.33±2.04         NaN

=== F1 (COVID) ===
                          0.63%        1.25%        12.5%         100%
variant                                                               
linear_probe        0.538±0.045  0.596±0.055  0.751±0.004  0.850±0.003
global_mlp          0.559±0.044  0.629±0.058  0.786±0.013  0.849±0

In [12]:
# CELL 11 - Phase 5: Final Write-up Scaffold (LaTeX Template Generation)
def generate_latex_template():
    tex_path = f"{DRIVE_DIR}/manuscript_scaffold.tex"
    content = r"""\documentclass{article}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{hyperref}

\title{Low-Data 3-Modality Architecture Pipelines for Medical Image Classification}
\author{Tasmia Tabassum, Humaira Khanam, Kazi Badrul Hasan Adrit \\ Computer Vision Lab (CVLab), Islamic University of Technology}
\date{\today}

\begin{document}
\maketitle

\begin{abstract}
We propose a robust multimodal classification architecture utilizing a global visual branch, a local dual-kernel bottleneck branch, and LLM-generated semantic text features, specifically optimized for extreme low-data regimes.
\end{abstract}

\section{Methodology}
% Discuss the integration of the ViT backbone with the custom 14x14 local DKA-style CNN block and BioVLM semantic features.

\section{Experiments and Evaluation}
% Insert findings from the CSV ablation tables generated by the training engine.
% Note the specific impacts at 0.63\%, 1.25\%, 12.5\%, and 100\% training fractions across both BUSI and COVID-19 datasets.

\section{Limitations \& Quality Audit}
% Reference the manual 50-caption audits and highlight where the text branch hallucinates or leaks diagnostic markers despite prompt strictness.

\end{document}
"""
    with open(tex_path, "w") as f:
        f.write(content)
    print(f"Saved LaTeX manuscript scaffold to {tex_path}")

generate_latex_template()

Saved LaTeX manuscript scaffold to ./mm_pipeline_covid/manuscript_scaffold.tex
